# NoClip · 2DGS 실험 (Google Colab)

줄무늬의 원인은 스플랫이 카메라 방향으로 길게 늘어나는 것. **2DGS** 는 스플랫을 납작한 원판으로 제한하고
깊이·법선 손실을 걸어서 이런 늘어남을 구조적으로 막음. 공식 구현 `hbb1/2d-gaussian-splatting` 을 받아서 학습함.

### 준비 (PC 에서 한 번)
`F:\Sample\6\1b\data` 처럼 **`images` 와 `sparse` 가 들어 있는 폴더**를 구글 드라이브에 올려 둠.
드라이브 안 위치는 상관없음 → 1번 셀의 `DRIVE_DATA` 를 그 폴더로 지정.

### 실행
1. **런타임 → 런타임 유형 변경 → T4 GPU** (L4 가 보이면 L4)
2. 1번 셀에서 `DRIVE_DATA`, `SCENE` 확인
3. **런타임 → 모두 실행**. 문제가 생기면 그 셀에서 멈추고 이유를 출력함

### 알아둘 점
- 결과는 드라이브 `3DGS/3_학습결과/4차_colab/out/<SCENE>/gs2d` 에 저장. `ply/` 안의 파일을 SuperSplat · Unity 에 넣으면 됨
- **런타임이 끊겨도 처음부터 다시 안 해도 됨.** 다시 "모두 실행" 하면 설치·복사는 다시 하지만, 학습은 마지막 체크포인트(5,000 스텝마다)부터 이어서 함
- 설치(CUDA 컴파일)는 런타임이 새로 켜질 때마다 다시 해야 함 (5~10분)
- 4K 사진을 그대로 읽으면 RAM(12GB)이 모자라서 끊김 → 로컬 복사본만 960 폭으로 줄여서 학습 (드라이브 원본은 그대로)
- 2DGS PLY 는 크기 값이 2개뿐이라 뷰어에서 바로 안 보임 → 마지막 셀이 3DGS 형식으로 변환
- 끝나면 **런타임 → 런타임 연결 해제 및 삭제** (무료 GPU 한도 절약)
- 라이선스: 원본 3DGS 와 같은 비상업 연구용

## 1. GPU 확인 · 드라이브 연결 · 설정

In [ ]:
import os, subprocess
if subprocess.run("nvidia-smi", shell=True, capture_output=True).returncode != 0:
    raise RuntimeError("GPU 런타임이 아님 -> 런타임 > 런타임 유형 변경 에서 T4/L4 선택. "
                       "선택지가 없으면 무료 GPU 한도 소진 상태이니 나중에 다시 시도")

from google.colab import drive
drive.mount('/content/drive')

# ── 여기만 바꾸면 됨 ──────────────────────────────────────────────
# images/ 와 sparse/ 가 들어 있는 드라이브 폴더 (예: 3DGS/2_정합데이터/4차/1 = 4차 촬영 구간 1)
DRIVE_DATA = "/content/drive/MyDrive/3DGS/2_정합데이터/4차/1"
SCENE      = "corridor1"     # 결과 폴더 이름
WIDTH      = 960             # 로컬 복사본을 이 폭으로 줄임 (4K 그대로면 RAM 초과)
SH         = 2               # Brush 와 동일
ITERS      = 30000
LAMBDA_NORMAL = 0.05         # 2DGS 기본값
LAMBDA_DIST   = 0.0          # 깊이 뭉침 벌점. 넓은 장면은 0 권장
# ─────────────────────────────────────────────────────────────────

DATA = f"/content/data/{SCENE}"                                   # 빠른 로컬 복사본
OUT  = f"/content/drive/MyDrive/3DGS/3_학습결과/4차_colab/out/{SCENE}/gs2d"
os.makedirs(OUT, exist_ok=True)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"                           # 상관없는 TensorFlow 경고 숨김

for sub in ["images", "sparse/0"]:
    if not os.path.isdir(f"{DRIVE_DATA}/{sub}"):
        raise RuntimeError(f"{DRIVE_DATA}/{sub} 가 없음. DRIVE_DATA 경로나 드라이브 업로드(동기화)를 확인. "
                           f"현재 내용: {os.listdir(DRIVE_DATA) if os.path.isdir(DRIVE_DATA) else '폴더 없음'}")
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True,
                     capture_output=True, text=True).stdout.strip())
print("사진", len(os.listdir(f"{DRIVE_DATA}/images")), "장 /", DRIVE_DATA)
print("결과 ->", OUT)

## 2. 설치 · CUDA 컴파일 (5~10분)
런타임이 새로 켜질 때마다 다시 필요함. 이미 설치돼 있으면 건너뜀.

In [ ]:
import os, subprocess
REPO = "/content/2d-gaussian-splatting"

def installed():
    return subprocess.run(["python", "-c", "import diff_surfel_rasterization, simple_knn"],
                          capture_output=True).returncode == 0

if installed():
    print("이미 설치됨 (건너뜀)")
else:
    if not os.path.exists(f"{REPO}/submodules/diff-surfel-rasterization/setup.py"):
        !rm -rf {REPO}
        !git clone -q --recursive https://github.com/hbb1/2d-gaussian-splatting.git {REPO}
    # simple-knn 은 gitlab 에서 받는데, 막히면 GitHub 미러로 대체
    if not os.path.exists(f"{REPO}/submodules/simple-knn/simple_knn.cu"):
        !rm -rf {REPO}/submodules/simple-knn
        !git clone -q https://github.com/camenduru/simple-knn.git {REPO}/submodules/simple-knn
    # 최신 컴파일러에서 나는 알려진 오류 방지 (이미 있으면 건너뜀)
    !grep -q "<cstdint>" {REPO}/submodules/diff-surfel-rasterization/cuda_rasterizer/rasterizer_impl.h || sed -i '1i #include <cstdint>' {REPO}/submodules/diff-surfel-rasterization/cuda_rasterizer/rasterizer_impl.h
    !grep -q "<cfloat>" {REPO}/submodules/simple-knn/simple_knn.cu || sed -i '1i #include <cfloat>' {REPO}/submodules/simple-knn/simple_knn.cu

    !pip install -q plyfile tqdm opencv-python lpips ninja
    os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5;8.9"     # T4 = 7.5, L4 = 8.9
    for mod in ["diff-surfel-rasterization", "simple-knn"]:
        print("컴파일:", mod)
        r = subprocess.run(["pip", "install", "--no-build-isolation", f"{REPO}/submodules/{mod}"],
                           capture_output=True, text=True)
        if r.returncode != 0:
            print((r.stdout + r.stderr)[-4000:])     # 실패 원인 (마지막 부분)
            raise RuntimeError(f"{mod} 컴파일 실패 - 위 출력의 error 부분을 확인")

if not installed():
    raise RuntimeError("설치 후에도 모듈을 못 불러옴")
# 새 PyTorch 는 torch.load 기본값이 바뀌어서 체크포인트 이어하기가 실패함 -> 예전 방식으로
!sed -i 's/torch.load(checkpoint)$/torch.load(checkpoint, weights_only=False)/' {REPO}/train.py
print("컴파일 OK")

## 3. 데이터 복사 (드라이브 → 로컬) · 사진 축소
카메라 시야각은 `sparse` 값으로 계산돼서 사진 크기만 줄여도 정합 결과는 그대로 쓸 수 있음.

In [ ]:
import os, shutil, glob, cv2
mark = f"{DATA}/.ready_{WIDTH}"
n_src = len(os.listdir(f"{DRIVE_DATA}/images"))

if os.path.exists(mark) and len(os.listdir(f"{DATA}/images")) == n_src:
    print("이미 준비됨 (건너뜀)")
else:
    shutil.rmtree(DATA, ignore_errors=True)
    os.makedirs(DATA)
    print("sparse 복사"); shutil.copytree(f"{DRIVE_DATA}/sparse", f"{DATA}/sparse")
    print("사진 복사 (드라이브에서 읽어서 수 분 걸림)")
    shutil.copytree(f"{DRIVE_DATA}/images", f"{DATA}/images")
    files = sorted(glob.glob(f"{DATA}/images/*"))
    for i, f in enumerate(files):
        im = cv2.imread(f)
        if im is None:
            raise RuntimeError(f"사진을 못 읽음: {f}")
        h, w = im.shape[:2]
        if w > WIDTH:
            im = cv2.resize(im, (WIDTH, round(h * WIDTH / w)), interpolation=cv2.INTER_AREA)
            cv2.imwrite(f, im, [cv2.IMWRITE_JPEG_QUALITY, 95])
        if i % 200 == 0:
            print(f"  축소 {i}/{len(files)}")
    print(f"사진 크기 {w}x{h} -> {im.shape[1]}x{im.shape[0]}")
    open(mark, "w").close()

print("sparse:", os.listdir(f"{DATA}/sparse"), "/ 사진:", len(os.listdir(f"{DATA}/images")), "장")

## 4. 2DGS 학습
5,000 스텝마다 체크포인트 저장 → 끊기면 다시 실행할 때 이어서 함. 10,000 / 20,000 / 마지막 스텝에 PLY 저장.

`Training progress` 진행바가 뜨면 정상. 처음 1~2분은 사진을 읽느라 멈춘 것처럼 보임.

In [ ]:
import os, glob, re
MODEL = f"{OUT}/model"
final = f"{MODEL}/point_cloud/iteration_{ITERS}/point_cloud.ply"

if os.path.exists(final):
    print("이미 학습 완료 (건너뜀):", final)
else:
    cks = sorted(glob.glob(f"{MODEL}/chkpnt*.pth"), key=lambda p: int(re.findall(r"chkpnt(\d+)", p)[-1]))
    resume = f'--start_checkpoint "{cks[-1]}"' if cks else ""
    print("이어서 학습:", cks[-1] if cks else "처음부터")
    ckpt_steps = " ".join(str(s) for s in range(5000, ITERS, 5000))
    cmd = (f'python train.py -s "{DATA}" -m "{MODEL}" -r 1 --sh_degree {SH} --iterations {ITERS} '
           f'--lambda_normal {LAMBDA_NORMAL} --lambda_dist {LAMBDA_DIST} --depth_ratio 0 '
           f'--save_iterations 10000 20000 {ITERS} --test_iterations -1 '
           f'--checkpoint_iterations {ckpt_steps} {resume}')
    !cd /content/2d-gaussian-splatting && {cmd}

if os.path.exists(final):
    for ck in glob.glob(f"{MODEL}/chkpnt*.pth"):    # 다 끝났으면 이어하기용 체크포인트(파일당 약 1GB)는 삭제
        os.remove(ck)
if not os.path.exists(final):
    raise RuntimeError("학습이 끝까지 안 됨. 위 출력 마지막 부분 확인. "
                       "'^C' 로 끝났으면 RAM 부족(WIDTH 를 줄이기) 또는 런타임 끊김 -> 다시 실행하면 이어서 함")
print("학습 완료")

## 5. PLY 변환 · 저장
2DGS 형식(크기 값 2개) → SuperSplat / Unity 가 읽는 3DGS 형식 (세 번째 크기를 거의 0 으로 채움).

In [ ]:
%%writefile /content/nc_ply2d_to_3d.py
import sys
import numpy as np

def read_ply(path):
    with open(path, "rb") as f:
        names, count = [], 0
        while True:
            line = f.readline().decode("ascii").strip()
            if line.startswith("element vertex"):
                count = int(line.split()[-1])
            elif line.startswith("property"):
                names.append(line.split()[-1])
            elif line == "end_header":
                break
        data = np.frombuffer(f.read(count * len(names) * 4), dtype="<f4").reshape(count, len(names))
    return names, data

src, dst = sys.argv[1], sys.argv[2]
thick = float(sys.argv[3]) if len(sys.argv) > 3 else -14.0    # log 값. exp(-14) = 8e-7
names, data = read_ply(src)
if "scale_2" in names:
    out_names, out = names, data
else:
    i = names.index("scale_1") + 1
    out_names = names[:i] + ["scale_2"] + names[i:]
    out = np.concatenate([data[:, :i], np.full((len(data), 1), thick, dtype="<f4"), data[:, i:]], axis=1)
with open(dst, "wb") as f:
    head = "ply\nformat binary_little_endian 1.0\nelement vertex %d\n" % len(out)
    head += "".join("property float %s\n" % n for n in out_names) + "end_header\n"
    f.write(head.encode("ascii"))
    f.write(np.ascontiguousarray(out, dtype="<f4").tobytes())
print(dst, f"{len(out):,} splats")

In [ ]:
import os, glob
os.makedirs(f"{OUT}/ply", exist_ok=True)
for it in sorted(glob.glob(f"{OUT}/model/point_cloud/iteration_*"), key=lambda p: int(p.split("_")[-1])):
    n = it.split("_")[-1]
    dst = f"{OUT}/ply/{SCENE}_2dgs_{n}.ply"
    if not os.path.exists(dst):
        !python /content/nc_ply2d_to_3d.py "{it}/point_cloud.ply" "{dst}"
!ls -la "{OUT}/ply"
print("\n끝. 런타임 > 런타임 연결 해제 및 삭제 로 GPU 를 반납할 것")